# Process NA DMS data from Wang et al.

In [1]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict
from Bio import SeqIO
import subprocess

In [2]:
import yaml

_config_dir = os.path.dirname(os.path.abspath("../config.yaml"))
with open("../config.yaml") as _f:
    _config = yaml.safe_load(_f)
data_dir = os.path.normpath(os.path.join(_config_dir, _config["data_dir"]))

In [3]:
# Read in the sequence of the reference NA used to build the N1 tree
ref_fasta = os.path.join(data_dir, 'NA/N1/curated_reference.fasta')
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate())
ref_aa_seq = ref_aa_seq.replace('*', '')
print('reference sequence length:', len(ref_aa_seq))

# Read in the DMS data
na_dms_data = pd.read_excel('../data/dms_data/Wang_NA/msystems.00670-23-s0006.xlsx')
na_dms_data['dms_site'] = na_dms_data['AA change'].apply(lambda x: int(x[1:-1]))
na_dms_data['wt_aa'] = na_dms_data['AA change'].apply(lambda x: x[0])
na_dms_data['mut_aa'] = na_dms_data['AA change'].apply(lambda x: x[-1])

# Get the sequence of the unmutated protein from the DMS experiment
dms_seq_df = (
    na_dms_data
    .sort_values('dms_site')
    .drop_duplicates('dms_site')
)
aa_seq = ''.join(list(dms_seq_df['wt_aa']))
print('DMS sequence length:', len(aa_seq))
print('DMS site range:', dms_seq_df['dms_site'].min(), '-', dms_seq_df['dms_site'].max())

# Save sequences to a FASTA file for alignment
output_dir = '../results/dms_data/Wang_NA/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
unaligned_fasta = os.path.join(output_dir, 'unaligned.fasta')
if not os.path.isfile(unaligned_fasta):
    with open(unaligned_fasta, 'w') as f:
        f.write('>reference\n')
        f.write(ref_aa_seq + '\n')
        f.write('>dms\n')
        f.write(aa_seq + '\n')

# Align the sequences using MUSCLE
aligned_fasta = os.path.join(output_dir, 'aligned.fasta')
if not os.path.isfile(aligned_fasta):
    cmd = ['muscle', '-align', unaligned_fasta, '-output', aligned_fasta]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)

reference sequence length: 454


DMS sequence length: 401
DMS site range: 7 - 407


In [4]:
# Read in the aligned sequences
seqs_dict = {}
aligned_records = list(SeqIO.parse(aligned_fasta, 'fasta'))
for record in aligned_records:
    seqs_dict[record.id] = str(record.seq)

aligned_ref_seq = seqs_dict['reference']
aligned_dms_seq = seqs_dict['dms']

# Compute percent identity
n_sites_to_compare = 0
n_identical = 0
for (alignment_index, (ref_aa, dms_aa)) in enumerate(zip(aligned_ref_seq, aligned_dms_seq), 1):
    if ref_aa != '-' and dms_aa != '-':
        n_sites_to_compare += 1
        if ref_aa == dms_aa:
            n_identical += 1

print(n_sites_to_compare, n_identical, n_identical / n_sites_to_compare)

# Determine numbering scheme
# DMS sequence is indexed starting at 7
dms_start_index = 7
numbering_dict = defaultdict(list)
assert len(aligned_ref_seq) == len(aligned_dms_seq)
for (seq_id, seq, start_index) in [
    ('tree_reference_site', aligned_ref_seq, 1),
    ('dms_site', aligned_dms_seq, dms_start_index),
]:
    seq_index = start_index
    for (alignment_index, aa) in enumerate(seq, 1):
        if aa != '-':
            numbering_dict['alignment_index'].append(alignment_index)
            numbering_dict['seq_id'].append(seq_id)
            numbering_dict['seq_index'].append(seq_index)
            numbering_dict['seq_aa'].append(aa)
            seq_index += 1

alignment_numbering_df = (
    pd.DataFrame(numbering_dict)
    .pivot(index='alignment_index', columns='seq_id', values='seq_index')
    .reset_index()
    .rename_axis(None, axis=1)
    .dropna()
    [['dms_site', 'tree_reference_site']]
)
alignment_numbering_df['dms_site'] = alignment_numbering_df['dms_site'].astype(int)
alignment_numbering_df['tree_reference_site'] = alignment_numbering_df['tree_reference_site'].astype(int)
alignment_numbering_df.head()

401 370 0.9226932668329177


,dms_site,tree_reference_site
6,7,7
7,8,8
8,9,9
9,10,10
10,11,11


In [5]:
# Merge DMS data with numbering scheme and save
na_dms_data_processed = (
    na_dms_data
    .merge(alignment_numbering_df, on='dms_site', validate='many_to_one')
)

# Compute the DMS effect as the log of the average of the three CTRL replicates
na_dms_data_processed['dms_effect'] = na_dms_data_processed[['RF_CTRL_Rep1', 'RF_CTRL_Rep2', 'RF_CTRL_Rep3']].mean(axis=1)
na_dms_data_processed = na_dms_data_processed[na_dms_data_processed['dms_effect'] > 0]
na_dms_data_processed['dms_effect'] = np.log(na_dms_data_processed['dms_effect'])

# Save data to output file
na_dms_data_processed = na_dms_data_processed[['dms_site', 'wt_aa', 'mut_aa', 'tree_reference_site', 'dms_effect']]
na_dms_data_processed.to_csv(os.path.join(output_dir, 'processed_dms_data.csv'), index=False)
print("Number of mutations with processed data:", len(na_dms_data_processed))
na_dms_data_processed.head()

Number of mutations with processed data: 2635


,dms_site,wt_aa,mut_aa,tree_reference_site,dms_effect
0,7,I,M,7,-1.468916
1,8,I,L,8,-2.304362
2,8,I,V,8,-0.997490
3,8,I,K,8,-2.152260
4,8,I,T,8,-0.425550
